# Targeted Attack Training for Demo Phrase (Demo Part 2: Injection)

This notebook demonstrates the targeted adversarial attack training for the live demo.

## Objective
- Record a generic voice sample
- Train Carlini-Wagner (CW) attack to inject specific phrase: "This is a Demo - aai590"
- Verify injection success and measure SNR
- Save trained perturbation for real-time demo

## Key Parameters
- Epsilon: 0.03-0.05 (imperceptibility constraint)
- Iterations: 1500-2500 (attack strength)
- Learning Rate: 0.01 (optimization rate)
- Target Sampling Rate: 16kHz (Whisper requirement)


In [ ]:
# Import required libraries
import os
import torch
import numpy as np
import soundfile as sf
import whisper
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Import attack modules
from src.attacks.cw import CarliniWagnerAttack
from src.models.whisper_wrapper import WhisperModelWrapper
from src.data.audio_loader import load_audio, save_audio

# Create demo assets directory
demo_assets_dir = Path('demo_assets')
demo_assets_dir.mkdir(exist_ok=True)

# Target phrase to inject
TARGET_PHRASE = "This is a Demo - aai590"

# CW Attack hyperparameters
CW_EPSILON = 0.04  # 0.03-0.05 range
CW_ITERATIONS = 2000  # 1500-2500 range
CW_LEARNING_RATE = 0.01
CW_C = 0.1  # Confidence parameter

## Step 1: Record Demo Voice Sample

Record a neutral voice sample for training the attack.

In [ ]:
def record_audio(duration=5, sample_rate=16000):
    """Simple audio recording using sounddevice."""
    import sounddevice as sd
    
    print(f"Recording for {duration} seconds... Please speak clearly.")
    recording = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
    sd.wait()
    
    # Normalize and clip to [-1, 1]
    recording = np.clip(recording / 32768.0, -1.0, 1.0)
    return recording

# Record demo audio (optional - can skip if using pre-recorded)
print("Recording voice sample...")
demo_audio = record_audio(duration=5, sample_rate=16000)
save_audio(demo_audio, demo_assets_dir / 'my_voice_generic.wav', sample_rate=16000)
print("Audio saved to demo_assets/my_voice_generic.wav")

## Step 2: Load Model and Preprocessing

In [ ]:
# Load Whisper model (small variant for faster training)
print("Loading Whisper model...")
whisper_model = whisper.load_model("base", device=device)

# Wrap for easier interface
model_wrapper = WhisperModelWrapper(whisper_model, device=device)

# Test on clean audio
print("Testing on clean audio...")
result = model_wrapper.transcribe(demo_audio, verbose=False)
print(f"Clean transcript: {result['text'].strip()}")

## Step 3: Train Carlini-Wagner Attack

Train the targeted attack to inject the specific phrase.

In [ ]:
def calculate_snr(original, perturbed):
    """Calculate Signal-to-Noise Ratio in dB."""
    # Convert to float64 for precision
    original = original.astype(np.float64)
    perturbed = perturbed.astype(np.float64)
    
    signal_power = np.mean(original ** 2)
    noise_power = np.mean((original - perturbed) ** 2)
    
    if noise_power == 0:
        return float('inf')
    
    return 10 * np.log10(signal_power / noise_power)

# Initialize CW attack
print("Initializing Carlini-Wagner attack...")
cw_attack = CarliniWagnerAttack(
    model=model_wrapper.model,
    device=device,
    epsilon=CW_EPSILON,
    max_iterations=CW_ITERATIONS,
    learning_rate=CW_LEARNING_RATE,
    c=CW_C,
    targeted=True
)

# Convert demo audio to tensor
demo_tensor = torch.from_numpy(demo_audio).unsqueeze(0).float().to(device)

# Target label (Whisper token IDs)
target_text = TARGET_PHRASE
target_tokens = whisper_model.encode(target_text)
print(f"Target phrase: '{target_text}'")
print(f"Target tokens: {target_tokens}")

# Run targeted attack
print(f"\nTraining targeted attack (iter {CW_ITERATIONS})...")
print(f"Target phrase: '{target_text}'")
print(f"Epsilon: {CW_EPSILON}, Learning Rate: {CW_LEARNING_RATE}")

with torch.no_grad():
    # Initialize perturbation
    perturbation = torch.zeros_like(demo_tensor)
    
    # Attack loop with progress tracking
    for i in tqdm(range(CW_ITERATIONS), desc="Training CW Attack"):
        # Forward pass
        adv_audio = demo_tensor + perturbation
        adv_audio = torch.clamp(adv_audio, -1.0, 1.0)
        
        # Get model output
        log_probs = model_wrapper.model(adv_audio, language="en", verbose=False)
        
        # Targeted loss: minimize negative log-probability of target
        target_loss = -log_probs[:, target_tokens[0]].mean()
        
        # Add confidence penalty to ensure target is chosen
        confidence_loss = log_probs.max(dim=1)[0].mean()
        
        # Combined loss
        loss = target_loss + CW_C * confidence_loss
        
        # Compute gradient and update
        loss.backward()
        
        # Update perturbation with clipping and epsilon constraint
        perturbation.data = torch.clamp(
            perturbation.data + CW_LEARNING_RATE * perturbation.grad.data,
            -CW_EPSILON, CW_EPSILON
        )
        
        # Zero gradients
        model_wrapper.model.zero_grad()
        
        # Log progress periodically
        if (i + 1) % 200 == 0:
            # Evaluate on current perturbation
            adv_audio_eval = torch.clamp(demo_tensor + perturbation, -1.0, 1.0)
            result = model_wrapper.transcribe(adv_audio_eval.cpu().numpy(), verbose=False)
            snr = calculate_snr(demo_audio, adv_audio_eval.cpu().numpy())
            print(f"\nIteration {i+1}:")
            print(f"  SNR: {snr:.2f} dB")
            print(f"  Transcript: '{result['text'].strip()}'")
            if TARGET_PHRASE in result['text'].lower():
                print("  ✓ Target phrase detected!")

In [ ]:
## Step 4: Final Evaluation

In [ ]:
# Generate final adversarial audio
with torch.no_grad():
    adv_audio_final = torch.clamp(demo_tensor + perturbation, -1.0, 1.0)
    adv_audio_final_cpu = adv_audio_final.cpu().numpy().squeeze()
    
# Evaluate results
clean_result = model_wrapper.transcribe(demo_audio, verbose=False)
adv_result = model_wrapper.transcribe(adv_audio_final_cpu, verbose=False)
snr = calculate_snr(demo_audio, adv_audio_final_cpu)

print("\n=== Final Evaluation ===")
print(f"Clean Transcript: {clean_result['text'].strip()}")
print(f"Adversarial Transcript: {adv_result['text'].strip()}")
print(f"SNR: {snr:.2f} dB")

# Save results
results = {
    'clean_transcript': clean_result['text'],
    'adversarial_transcript': adv_result['text'],
    'target_phrase': target_text,
    'snr': float(snr),
    'epsilon': CW_EPSILON,
    'iterations': CW_ITERATIONS
}

import json
with open(demo_assets_dir / 'targeted_attack_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nResults saved to demo_assets/targeted_attack_results.json")

## Step 5: Save Trained Perturbation

In [ ]:
# Save the perturbation tensor
torch.save(perturbation.cpu(), demo_assets_dir / 'targeted_perturbation.pt')
print(f"Trained perturbation saved to demo_assets/targeted_perturbation.pt")

# Save the adversarial audio for playback testing
save_audio(adv_audio_final_cpu, demo_assets_dir / 'targeted_adversarial.wav', sample_rate=16000)
print(f"Adversarial audio saved to demo_assets/targeted_adversarial.wav")

# Display waveform comparison
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(demo_audio, color='blue', alpha=0.6)
axes[0].set_title('Original Audio')
axes[0].set_xlabel('Samples')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

axes[1].plot(adv_audio_final_cpu, color='red', alpha=0.6, label='Adversarial')
axes[1].set_title(f'Adversarial Audio (SNR: {snr:.2f} dB)')
axes[1].set_xlabel('Samples')
axes[1].set_ylabel('Amplitude')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig(demo_assets_dir / 'waveform_comparison.png', dpi=100)
print(f"Waveform comparison saved to demo_assets/waveform_comparison.png")

print("\n=== Training Complete ===")
print("You can now use the perturbation in the live demo!")

## Troubleshooting Tips

1. **Low SNR**: Try increasing epsilon (0.03-0.05) or iterations (1500-2500)
2. **Attack Failure**: Ensure the target phrase is valid in Whisper's vocabulary
3. **Audio Clipping**: Check that epsilon constraint is enforced properly
4. **GPU Memory**: Reduce model size (use 'tiny' or 'base' instead of 'small')